# 04 — Model Evaluation

Comprehensive evaluation of the trained YOLOv8n bib-detection model on the held-out test split (21 images).  
Covers quantitative metrics, PR / confusion-matrix artefacts, per-image failure analysis, and a summary table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from ultralytics import YOLO

DEVICE   = "mps" if torch.backends.mps.is_available() else "cpu"
ROOT     = Path("..")
RUN_DIR  = ROOT / "models" / "runs" / "bib-yolov8n"
DATA_YAML= ROOT / "data" / "labeled" / "race-vision.v1i.yolov8" / "data.yaml"
TEST_IMG = ROOT / "data" / "labeled" / "race-vision.v1i.yolov8" / "test" / "images"
TEST_LBL = ROOT / "data" / "labeled" / "race-vision.v1i.yolov8" / "test" / "labels"

WEIGHTS  = RUN_DIR / "weights" / "best.pt"
CONF_THR = 0.25   # detection confidence threshold
IOU_THR  = 0.50   # IoU threshold for TP matching

print(f"device : {DEVICE}")
print(f"weights: {WEIGHTS.resolve()}")
print(f"test images: {len(list(TEST_IMG.glob('*')))}")

## 1. Load Model

In [ ]:
model = YOLO(WEIGHTS)
print(model.info())

## 2. Test-Set Metrics

`model.val(split="test")` re-runs inference on the 21 test images and computes
precision, recall, mAP50, and mAP50-95 against ground-truth labels.

In [ ]:
metrics = model.val(
    data=str(DATA_YAML.resolve()),
    split="test",
    conf=CONF_THR,
    iou=IOU_THR,
    device=DEVICE,
    verbose=False,
)

p   = metrics.box.mp
r   = metrics.box.mr
m50 = metrics.box.map50
m   = metrics.box.map

summary = pd.DataFrame([{
    "Precision": f"{p:.3f}",
    "Recall":    f"{r:.3f}",
    "mAP50":     f"{m50:.3f}",
    "mAP50-95":  f"{m:.3f}",
}])
summary.index = ["bib"]
print("\nTest-set results")
display(summary)

## 3. Training History

In [ ]:
df = pd.read_csv(RUN_DIR / "results.csv", skipinitialspace=True)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df["epoch"], df["train/box_loss"], label="train")
axes[0].plot(df["epoch"], df["val/box_loss"],   label="val")
axes[0].set_title("Box Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
axes[1].plot(df["epoch"], df["metrics/recall(B)"],    label="Recall")
axes[1].set_title("Precision & Recall")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(df["epoch"], df["metrics/mAP50(B)"],    label="mAP50")
axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[2].set_title("mAP")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.show()

best_row = df.loc[df["metrics/mAP50(B)"].idxmax()]
print(f"Best val mAP50 {best_row['metrics/mAP50(B)']:.3f}  "
      f"mAP50-95 {best_row['metrics/mAP50-95(B)']:.3f}  "
      f"@ epoch {int(best_row['epoch'])}")

## 4. Confusion Matrix & PR Curve

Ultralytics saves these artefacts during `val()`; we display them here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_path = RUN_DIR / "confusion_matrix_normalized.png"
pr_path = RUN_DIR / "BoxPR_curve.png"

for ax, path, title in zip(axes, [cm_path, pr_path],
                            ["Confusion Matrix (normalised)", "Precision-Recall Curve"]):
    if path.exists():
        ax.imshow(Image.open(path))
        ax.set_title(title, fontsize=13)
    else:
        ax.text(0.5, 0.5, f"{path.name}\nnot found", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.savefig("cm_pr.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Per-Image Inference on Test Split

Run the model on every test image and collect predicted boxes, confidences, and ground-truth boxes.

In [ ]:
def parse_gt(label_path: Path, img_w: int, img_h: int) -> np.ndarray:
    """Return GT boxes as pixel [x1, y1, x2, y2] array."""
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cx, cy, w, h = map(float, parts[1:5])
            boxes.append([
                (cx - w / 2) * img_w, (cy - h / 2) * img_h,
                (cx + w / 2) * img_w, (cy + h / 2) * img_h,
            ])
    return np.array(boxes) if boxes else np.zeros((0, 4))


def box_iou(b1: np.ndarray, b2: np.ndarray) -> float:
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return float(inter / union) if union > 0 else 0.0


img_paths = sorted(TEST_IMG.glob("*"))
records   = []

for img_path in img_paths:
    img = Image.open(img_path)
    W, H = img.size

    lbl_path = TEST_LBL / (img_path.stem + ".txt")
    gt_boxes = parse_gt(lbl_path, W, H)

    res  = model.predict(source=str(img_path), conf=CONF_THR, device=DEVICE, verbose=False)
    pred = res[0].boxes
    pred_boxes = pred.xyxy.cpu().numpy() if len(pred) else np.zeros((0, 4))
    pred_confs = pred.conf.cpu().numpy() if len(pred) else np.array([])

    records.append({
        "path":       img_path,
        "img":        img,
        "gt_boxes":   gt_boxes,
        "pred_boxes": pred_boxes,
        "pred_confs": pred_confs,
        "n_gt":       len(gt_boxes),
        "n_pred":     len(pred_boxes),
    })

print(f"Processed {len(records)} test images")
print(f"Total GT boxes  : {sum(r['n_gt']   for r in records)}")
print(f"Total detections: {sum(r['n_pred'] for r in records)}")

## 6. Match Predictions → Ground Truth

Greedy IoU matching at threshold 0.5 to categorise each detection as TP, FP, or FN.

In [ ]:
def match_boxes(pred_boxes, gt_boxes, iou_thr=IOU_THR):
    """Return (tp_pred_idx, fp_pred_idx, fn_gt_idx)."""
    matched_gt  = set()
    tp_pred, fp_pred = [], []

    for pi, pb in enumerate(pred_boxes):
        best_iou, best_gi = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            v = box_iou(pb, gb)
            if v > best_iou:
                best_iou, best_gi = v, gi
        if best_iou >= iou_thr:
            tp_pred.append(pi)
            matched_gt.add(best_gi)
        else:
            fp_pred.append(pi)

    fn_gt = [gi for gi in range(len(gt_boxes)) if gi not in matched_gt]
    return tp_pred, fp_pred, fn_gt


total_tp = total_fp = total_fn = 0
for r in records:
    tp, fp, fn = match_boxes(r["pred_boxes"], r["gt_boxes"])
    r["tp"], r["fp"], r["fn"] = tp, fp, fn
    total_tp += len(tp)
    total_fp += len(fp)
    total_fn += len(fn)

prec           = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
matched_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1             = 2 * prec * matched_recall / (prec + matched_recall) if (prec + matched_recall) else 0

print(f"TP {total_tp}  FP {total_fp}  FN {total_fn}")
print(f"Precision {prec:.3f}  Recall {matched_recall:.3f}  F1 {f1:.3f}  (conf≥{CONF_THR}, IoU≥{IOU_THR})")

## 7. Sample Predictions (TP / FP / FN overlay)

In [ ]:
def draw_record(ax, record):
    ax.imshow(record["img"])
    for gi, gb in enumerate(record["gt_boxes"]):
        color = "lime" if gi not in record["fn"] else "red"
        rect  = mpatches.Rectangle(
            (gb[0], gb[1]), gb[2] - gb[0], gb[3] - gb[1],
            linewidth=2, edgecolor=color, facecolor="none", linestyle="--"
        )
        ax.add_patch(rect)
    for pi, pb in enumerate(record["pred_boxes"]):
        color = "cyan" if pi in record["tp"] else "orange"
        rect  = mpatches.Rectangle(
            (pb[0], pb[1]), pb[2] - pb[0], pb[3] - pb[1],
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(pb[0], pb[1] - 4, f"{record['pred_confs'][pi]:.2f}",
                color=color, fontsize=7, fontweight="bold")
    n_tp = len(record["tp"])
    n_fp = len(record["fp"])
    n_fn = len(record["fn"])
    ax.set_title(f"TP {n_tp}  FP {n_fp}  FN {n_fn}", fontsize=9)
    ax.axis("off")


n_show = min(9, len(records))
cols = 3
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 5))
axes = axes.flatten()

for i, rec in enumerate(records[:n_show]):
    draw_record(axes[i], rec)

for j in range(n_show, len(axes)):
    axes[j].axis("off")

legend = [
    mpatches.Patch(edgecolor="lime",   facecolor="none", linestyle="--", label="GT (matched)"),
    mpatches.Patch(edgecolor="red",    facecolor="none", linestyle="--", label="GT (missed / FN)"),
    mpatches.Patch(edgecolor="cyan",   facecolor="none", label="Pred TP"),
    mpatches.Patch(edgecolor="orange", facecolor="none", label="Pred FP"),
]
fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=10, frameon=True)

plt.suptitle("Test-set predictions (first 9 images)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("sample_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Failure Analysis

### 8a. False Positives — detections with no matching GT bib

In [ ]:
fp_cases = [(r, pi) for r in records for pi in r["fp"]]
print(f"{len(fp_cases)} false-positive detections across test set")

if fp_cases:
    n_show = min(6, len(fp_cases))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for idx, (rec, pi) in enumerate(fp_cases[:n_show]):
        pb   = rec["pred_boxes"][pi]
        conf = rec["pred_confs"][pi]
        # crop around the FP with 50 px padding
        W, H = rec["img"].size
        x1 = max(0, int(pb[0]) - 50)
        y1 = max(0, int(pb[1]) - 50)
        x2 = min(W, int(pb[2]) + 50)
        y2 = min(H, int(pb[3]) + 50)
        crop = rec["img"].crop((x1, y1, x2, y2))
        axes[idx].imshow(crop)
        rect = mpatches.Rectangle(
            (pb[0] - x1, pb[1] - y1), pb[2] - pb[0], pb[3] - pb[1],
            linewidth=2, edgecolor="orange", facecolor="none"
        )
        axes[idx].add_patch(rect)
        axes[idx].set_title(f"{rec['path'].name}\nconf={conf:.2f}", fontsize=8)
        axes[idx].axis("off")

    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    plt.suptitle("False Positives (crops)", fontsize=13)
    plt.tight_layout()
    plt.savefig("false_positives.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No false positives — nothing to display.")

### 8b. False Negatives — missed GT bibs

In [ ]:
fn_cases = [(r, gi) for r in records for gi in r["fn"]]
print(f"{len(fn_cases)} false-negative (missed) GT boxes across test set")

if fn_cases:
    n_show = min(6, len(fn_cases))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for idx, (rec, gi) in enumerate(fn_cases[:n_show]):
        gb = rec["gt_boxes"][gi]
        W, H = rec["img"].size
        x1 = max(0, int(gb[0]) - 50)
        y1 = max(0, int(gb[1]) - 50)
        x2 = min(W, int(gb[2]) + 50)
        y2 = min(H, int(gb[3]) + 50)
        crop = rec["img"].crop((x1, y1, x2, y2))
        axes[idx].imshow(crop)
        rect = mpatches.Rectangle(
            (gb[0] - x1, gb[1] - y1), gb[2] - gb[0], gb[3] - gb[1],
            linewidth=2, edgecolor="red", facecolor="none", linestyle="--"
        )
        axes[idx].add_patch(rect)
        axes[idx].set_title(f"{rec['path'].name}", fontsize=8)
        axes[idx].axis("off")

    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    plt.suptitle("False Negatives — missed GT bibs (crops)", fontsize=13)
    plt.tight_layout()
    plt.savefig("false_negatives.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No false negatives — all GT bibs were detected.")

## 9. Confidence Distribution

In [ ]:
tp_confs = [records[ri]["pred_confs"][pi]
            for ri, rec in enumerate(records)
            for pi in rec["tp"]]
fp_confs = [records[ri]["pred_confs"][pi]
            for ri, rec in enumerate(records)
            for pi in rec["fp"]]

bins = np.linspace(0, 1, 21)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tp_confs, bins=bins, alpha=0.7, label=f"TP (n={len(tp_confs)})", color="steelblue")
ax.hist(fp_confs, bins=bins, alpha=0.7, label=f"FP (n={len(fp_confs)})", color="tomato")
ax.axvline(CONF_THR, color="black", linestyle=":", label=f"conf thr={CONF_THR}")
ax.set_xlabel("Confidence score")
ax.set_ylabel("Count")
ax.set_title("Confidence distribution — TP vs FP detections")
ax.legend()
plt.tight_layout()
plt.savefig("confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

if tp_confs:
    print(f"TP conf  mean={np.mean(tp_confs):.3f}  min={np.min(tp_confs):.3f}")
if fp_confs:
    print(f"FP conf  mean={np.mean(fp_confs):.3f}  min={np.min(fp_confs):.3f}")

## 10. Box-Size Analysis

Examine whether the model struggles with small bibs (common in race photography where runners are far from camera).

In [ ]:
def box_area_frac(box, img):
    W, H = img.size
    return ((box[2] - box[0]) * (box[3] - box[1])) / (W * H)

tp_sizes, fp_sizes, fn_sizes = [], [], []
for rec in records:
    for pi in rec["tp"]:
        tp_sizes.append(box_area_frac(rec["pred_boxes"][pi], rec["img"]))
    for pi in rec["fp"]:
        fp_sizes.append(box_area_frac(rec["pred_boxes"][pi], rec["img"]))
    for gi in rec["fn"]:
        fn_sizes.append(box_area_frac(rec["gt_boxes"][gi], rec["img"]))

bins = np.logspace(-4, -0.5, 20)
fig, ax = plt.subplots(figsize=(9, 4))
for data, label, color in [
    (tp_sizes, f"TP (n={len(tp_sizes)})",  "steelblue"),
    (fp_sizes, f"FP (n={len(fp_sizes)})",  "tomato"),
    (fn_sizes, f"FN (n={len(fn_sizes)})",  "gold"),
]:
    if data:
        ax.hist(data, bins=bins, alpha=0.6, label=label, color=color)
ax.set_xscale("log")
ax.set_xlabel("Box area / image area (log scale)")
ax.set_ylabel("Count")
ax.set_title("Box-size distribution by outcome")
ax.legend()
plt.tight_layout()
plt.savefig("box_size_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Summary

In [ ]:
best_row = df.loc[df["metrics/mAP50(B)"].idxmax()]

rows = [
    ("Model",               "YOLOv8n"),
    ("Test images",          str(len(records))),
    ("GT boxes",             str(sum(r['n_gt']   for r in records))),
    ("Detections",           str(sum(r['n_pred'] for r in records))),
    ("TP / FP / FN",         f"{total_tp} / {total_fp} / {total_fn}"),
    ("Precision  (matched)", f"{prec:.3f}"),
    ("Recall     (matched)", f"{matched_recall:.3f}"),
    ("F1         (matched)", f"{f1:.3f}"),
    ("mAP50  (YOLO val)",    f"{m50:.3f}"),
    ("mAP50-95 (YOLO val)",  f"{m:.3f}"),
    ("Best val mAP50 (train)", f"{best_row['metrics/mAP50(B)']:.3f}  @ epoch {int(best_row['epoch'])}"),
]

df_summary = pd.DataFrame(rows, columns=["Metric", "Value"]).set_index("Metric")
display(df_summary)